[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/C.%20Transportation%2C%20Routing%20%26%20Freight%20Management%20%28The%20Physical%20Internet%29/074.%20The%20VRP%20with%20Backhauls%20%28VRPB%29/P74-Tier-5_executed.ipynb)

---


# 74. The Vehicle Routing Problem with Backhauls

## Tier 5: A Digital-Twin-Inspired Telemetry Simulation

### Purpose

This notebook connects a small Vehicle Routing Problem with Backhauls (VRPB) instance to a synthetic telemetry and traffic-disruption simulation. It demonstrates how sensor observations can be collected and how a simple threshold rule can flag low traffic speeds. It does **not** compute VRPB routes, predict disruptions, or reoptimize routes.

The distinction matters: a digital twin is a virtual representation connected to an operational system through data and models. This notebook is a small, self-contained simulation of selected data and detection components, not a deployed or fully integrated digital twin.

### Inputs and outputs

- **Inputs:** a fixed VRPB example, a synthetic sensor generator, and a disruption threshold.
- **Outputs:** generated sensor records, traffic-disruption events identified by the threshold rule, summary counts, and visualizations of the simulated data.
- **Reproducibility:** sensor values use NumPy's random seed `42`. Simulation timestamps use the current clock, so timestamps and timestamp-based event identifiers are not fixed across runs.

### VRPB background

A VRPB has a depot, delivery customers, pickup (backhaul) customers, and vehicles with limited capacity. A vehicle leaves the depot carrying goods for delivery customers. After serving deliveries, it collects goods from pickup customers and returns to the depot. The defining precedence condition is that deliveries assigned to a route are served before pickups on that route. The `distance_matrix` and demand data below describe a small instance; this notebook does not use them to construct or optimize routes.

### Modeling assumptions and scope

- The sensor stream is synthetic. Each simulated minute generates seven traffic readings and three vehicle readings; it does not represent live IoT data.
- The configuration value of 50,000 data points per hour is a target specification, not the rate generated by this sample simulation.
- A traffic reading below 20 km/h is assessed using a severity score. An event is recorded only when its severity exceeds the configured threshold.
- Estimated event duration and predicted impact are simple illustrative formulas, not forecasts learned from data or validated against operations.
- No route assignments, delivery-before-pickup checks, rerouting decisions, cost savings, or delay reductions are calculated. Warehouse, customer, and driver systems are not simulated.

The implementation follows this sequence: define the instance and configuration, generate synthetic sensor observations, detect threshold-based traffic events, run the simulation, and inspect the observed data.

## 1. Imports and visualization setup

The notebook uses NumPy for the distance matrix and synthetic measurements, Matplotlib and Seaborn for plotting, and dataclasses to keep the instance, sensor records, events, and configuration explicit. The simulation is deliberately kept separate from any route optimization procedure.

In [ ]:
from dataclasses import dataclass
from datetime import datetime, timedelta
from typing import Any, Dict, List, Optional
from collections import deque

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

## 2. Data structures

`VRPBInstance` stores the routing data and creates customer indices: index 0 is the depot, delivery indices follow it, and pickup indices follow the deliveries. The class checks basic data integrity so that the distance matrix and demand lists match the declared instance.

`SensorData` represents one observation. `DisruptionEvent` records the output of the threshold detector. `DigitalTwinConfig` separates configuration values from the simulation logic. The prediction horizon and update frequency are configuration fields only; this notebook does not implement prediction or periodic system updates.

In [ ]:
@dataclass
class VRPBInstance:
    """Data for a VRP with Backhauls instance."""

    num_deliveries: int
    num_pickups: int
    delivery_demands: List[float]
    pickup_demands: List[float]
    capacity: float
    distance_matrix: np.ndarray

    def __post_init__(self) -> None:
        if self.num_deliveries < 0 or self.num_pickups < 0:
            raise ValueError("Customer counts must be nonnegative.")
        if len(self.delivery_demands) != self.num_deliveries:
            raise ValueError("Delivery demand count must match num_deliveries.")
        if len(self.pickup_demands) != self.num_pickups:
            raise ValueError("Pickup demand count must match num_pickups.")
        if self.capacity <= 0:
            raise ValueError("Vehicle capacity must be positive.")

        expected_size = 1 + self.num_deliveries + self.num_pickups
        if self.distance_matrix.shape != (expected_size, expected_size):
            raise ValueError("Distance matrix dimensions must match the depot and customers.")
        if np.any(self.distance_matrix < 0):
            raise ValueError("Distances must be nonnegative.")
        if not np.allclose(self.distance_matrix, self.distance_matrix.T):
            raise ValueError("This example uses a symmetric distance matrix.")

        self.num_customers = self.num_deliveries + self.num_pickups
        self.delivery_indices = list(range(1, self.num_deliveries + 1))
        self.pickup_indices = list(
            range(self.num_deliveries + 1, self.num_customers + 1)
        )
        self.all_customer_indices = self.delivery_indices + self.pickup_indices


@dataclass
class SensorData:
    """One synthetic observation from a simulated sensor."""

    timestamp: datetime
    sensor_id: str
    sensor_type: str
    location_id: int
    data_value: float
    unit: str
    quality_score: float


@dataclass
class DisruptionEvent:
    """A traffic event identified by the threshold-based detector."""

    event_id: str
    event_type: str
    location: int
    severity: float
    estimated_duration: float
    affected_routes: List[int]
    detection_time: datetime
    predicted_impact: float


@dataclass
class DigitalTwinConfig:
    """Configuration values for the telemetry simulation."""

    data_points_per_hour: int = 50000
    prediction_horizon: int = 120
    update_frequency: int = 30
    disruption_threshold: float = 0.3
    optimization_frequency: int = 300

## 3. Create the VRPB example

The distance matrix includes the depot at index 0, four delivery customers, and three pickup customers. Its entries are distances in the example's distance units. Delivery and pickup demands are quantities in the same capacity units as the vehicle capacity. These data define the routing context; the telemetry simulation below does not calculate a route or use the distances.

In [ ]:
def create_concrete_example() -> VRPBInstance:
    """Create the fixed VRPB instance used as the routing context."""
    num_deliveries = 4
    num_pickups = 3
    delivery_demands = [5, 4, 6, 3]
    pickup_demands = [4, 5, 3]
    capacity = 20

    distance_matrix = np.array(
        [
            [0, 3, 4, 5, 6, 8, 7, 9],
            [3, 0, 2, 3, 4, 6, 5, 7],
            [4, 2, 0, 2, 3, 5, 4, 6],
            [5, 3, 2, 0, 2, 4, 3, 5],
            [6, 4, 3, 2, 0, 3, 2, 4],
            [8, 6, 5, 4, 3, 0, 2, 3],
            [7, 5, 4, 3, 2, 2, 0, 2],
            [9, 7, 6, 5, 4, 3, 2, 0],
        ]
    )

    return VRPBInstance(
        num_deliveries=num_deliveries,
        num_pickups=num_pickups,
        delivery_demands=delivery_demands,
        pickup_demands=pickup_demands,
        capacity=capacity,
        distance_matrix=distance_matrix,
    )


instance = create_concrete_example()
print(f"Deliveries: {instance.num_deliveries}")
print(f"Pickups: {instance.num_pickups}")
print(f"Vehicle capacity: {instance.capacity}")
print(f"Total delivery demand: {sum(instance.delivery_demands)}")
print(f"Total pickup demand: {sum(instance.pickup_demands)}")

## 4. Generate synthetic sensor observations

For each simulated minute, the generator produces one traffic reading for each of seven locations and one vehicle reading for each of three vehicles. Thus, each batch contains ten observations. The configured hourly data target does not control this sample generator; the generated sample rate is determined by the loops below.

The fixed NumPy seed makes the sequence of generated sensor values reproducible when this notebook is run from the top. The timestamps still use the simulation's start time.

In [ ]:
class IoTDataGenerator:
    """Generate synthetic traffic and vehicle sensor readings."""

    def __init__(self, config: DigitalTwinConfig):
        self.config = config
        np.random.seed(42)

    def generate_sensor_batch(self, current_time: datetime) -> List[SensorData]:
        """Generate one minute's sample of sensor data."""
        sensor_data = []

        for location in range(1, 8):
            traffic_speed = np.random.uniform(10, 60)
            sensor_data.append(
                SensorData(
                    timestamp=current_time,
                    sensor_id=f"traffic_{location}",
                    sensor_type="traffic",
                    location_id=location,
                    data_value=traffic_speed,
                    unit="km/h",
                    quality_score=0.9,
                )
            )

        for vehicle_id in range(3):
            fuel_efficiency = np.random.uniform(8, 12)
            sensor_data.append(
                SensorData(
                    timestamp=current_time,
                    sensor_id=f"vehicle_{vehicle_id}",
                    sensor_type="vehicle",
                    location_id=vehicle_id,
                    data_value=fuel_efficiency,
                    unit="km/l",
                    quality_score=0.95,
                )
            )

        return sensor_data

## 5. Detect threshold-based traffic disruptions

The detector first checks whether the observed speed is below 20 km/h. For such an observation, it computes a severity score and compares that score with the configured threshold:

$$
s(v) = 1 - \frac{v}{20}, \qquad
\text{record an event if } v < 20 \text{ and } s(v) > \tau.
$$

Here, $v$ is the measured traffic speed in km/h, $s(v)$ is a dimensionless severity score, and $\tau$ is `disruption_threshold`. The code assigns an illustrative estimated duration of $60s(v)$ minutes and an illustrative predicted impact of $30s(v)$ minutes. These formulas are transparent rules, not empirically calibrated prediction models. The detector has no route data, so `affected_routes` remains empty.

In [ ]:
class DisruptionDetector:
    """Flag low-speed traffic readings using a severity threshold."""

    def __init__(self, config: DigitalTwinConfig):
        self.config = config
        self.disruption_history: List[DisruptionEvent] = []

    def detect_traffic_disruption(
        self, traffic_data: SensorData
    ) -> Optional[DisruptionEvent]:
        """Return an event when a traffic reading exceeds the severity threshold."""
        if traffic_data.sensor_type != "traffic":
            return None

        if traffic_data.data_value >= 20:
            return None

        severity = 1.0 - (traffic_data.data_value / 20.0)
        if severity <= self.config.disruption_threshold:
            return None

        disruption = DisruptionEvent(
            event_id=(
                f"traffic_{traffic_data.location_id}_"
                f"{traffic_data.timestamp.isoformat()}"
            ),
            event_type="traffic",
            location=traffic_data.location_id,
            severity=severity,
            estimated_duration=severity * 60.0,
            affected_routes=[],
            detection_time=traffic_data.timestamp,
            predicted_impact=severity * 30.0,
        )
        self.disruption_history.append(disruption)
        return disruption

## 6. Assemble the simulation

`DigitalTwinSystem` connects the generator and detector. During each simulated minute, it stores the generated sample, checks traffic readings, and records any detected events. The sensor buffer is a bounded `deque`: it retains recent observations up to its capacity. Event history and per-minute metrics are stored separately.

For a run of $T$ minutes, the generator creates $10T$ observations and the detector checks the seven traffic observations in each batch. The processing time therefore grows linearly with the number of simulated minutes, $O(T)$. Memory for the sensor buffer is bounded by its configured capacity; event and metric history grows with the duration of the run.

In [ ]:
class DigitalTwinSystem:
    """Coordinate synthetic telemetry generation and event detection."""

    def __init__(self, instance: VRPBInstance):
        self.instance = instance
        self.config = DigitalTwinConfig()
        self.data_generator = IoTDataGenerator(self.config)
        self.disruption_detector = DisruptionDetector(self.config)
        self.sensor_data_stream = deque(maxlen=1000)
        self.disruption_log: List[DisruptionEvent] = []
        self.system_metrics: List[Dict[str, Any]] = []

    def simulate_real_time_operation(
        self, duration_minutes: int = 60
    ) -> Dict[str, Any]:
        """Simulate minute-by-minute telemetry collection and event detection."""
        if duration_minutes < 0:
            raise ValueError("Simulation duration must be nonnegative.")

        start_time = datetime.now()
        simulation_results = {
            "duration_minutes": duration_minutes,
            "data_points_processed": 0,
            "disruptions_detected": 0,
        }

        for minute in range(duration_minutes):
            current_time = start_time + timedelta(minutes=minute)
            sensor_data = self.data_generator.generate_sensor_batch(current_time)
            self.sensor_data_stream.extend(sensor_data)
            simulation_results["data_points_processed"] += len(sensor_data)

            disruptions = self._detect_disruptions(sensor_data)
            self.disruption_log.extend(disruptions)
            simulation_results["disruptions_detected"] += len(disruptions)

            self.system_metrics.append(
                {
                    "timestamp": current_time,
                    "data_points_processed": len(sensor_data),
                    "disruptions_detected": len(disruptions),
                }
            )

        return simulation_results

    def _detect_disruptions(
        self, sensor_data: List[SensorData]
    ) -> List[DisruptionEvent]:
        """Apply the detector to traffic readings in one generated batch."""
        disruptions = []
        for data in sensor_data:
            if data.sensor_type == "traffic":
                disruption = self.disruption_detector.detect_traffic_disruption(data)
                if disruption is not None:
                    disruptions.append(disruption)
        return disruptions

## 7. Inspect the initial configuration

This cell creates the system and displays its configured values. The hourly data-point figure is a target specification; the later simulation summary reports the observations actually generated by this sample implementation.

In [ ]:
digital_twin = DigitalTwinSystem(instance)

print("Initial telemetry simulation configuration")
print(f"Customers in VRPB instance: {instance.num_customers}")
print(f"Configured data target: {digital_twin.config.data_points_per_hour:,} points/hour")
print(f"Prediction horizon setting: {digital_twin.config.prediction_horizon} minutes")
print(f"Update frequency setting: {digital_twin.config.update_frequency} seconds")
print(f"Traffic disruption threshold: {digital_twin.config.disruption_threshold}")
print("Generated sample size: 7 traffic readings and 3 vehicle readings per simulated minute")

## 8. Run the simulation and inspect the summary

The reported counts come directly from the simulation. A detected event means only that a synthetic traffic reading passed the stated threshold rule. It does not imply that a route was changed or that an operational delay or financial saving was achieved.

In [ ]:
simulation_results = digital_twin.simulate_real_time_operation(duration_minutes=60)

print("Telemetry simulation results")
for key, value in simulation_results.items():
    print(f"{key.replace('_', ' ').title()}: {value}")

if simulation_results["duration_minutes"] > 0:
    observed_rate = (
        simulation_results["data_points_processed"]
        / simulation_results["duration_minutes"]
    )
    print(f"Generated sample rate: {observed_rate:.1f} data points per simulated minute")
else:
    print("Generated sample rate: not defined for a zero-minute run")

print(
    "Sensor observations retained in buffer: "
    f"{len(digital_twin.sensor_data_stream)}"
)

## 9. Visualize the generated telemetry and detected events

Each panel uses observations retained by the simulation, rather than fixed illustrative values. The bounded sensor buffer can limit what is available for plotting in longer runs. The traffic-speed panel distinguishes locations by color, and the severity panel displays the detector threshold.

In [ ]:
def visualize_digital_twin_performance(
    digital_twin: DigitalTwinSystem,
    simulation_results: Dict[str, Any],
) -> None:
    """Plot sensor counts, traffic readings, and detected events."""
    sensor_records = list(digital_twin.sensor_data_stream)
    sensor_frame = pd.DataFrame([vars(record) for record in sensor_records])

    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle("Synthetic Telemetry Simulation", fontsize=16, fontweight="bold")

    if sensor_frame.empty:
        for axis in axes.flat:
            axis.text(
                0.5,
                0.5,
                "No sensor observations are available",
                ha="center",
                va="center",
                transform=axis.transAxes,
            )
            axis.set_axis_off()
        plt.tight_layout()
        plt.show()
        return

    sensor_counts = sensor_frame["sensor_type"].value_counts().sort_index()
    axes[0, 0].bar(sensor_counts.index, sensor_counts.values)
    axes[0, 0].set_title("Retained observations by sensor type")
    axes[0, 0].set_xlabel("Sensor type")
    axes[0, 0].set_ylabel("Observation count")

    traffic_frame = sensor_frame[sensor_frame["sensor_type"] == "traffic"]
    for location_id, location_data in traffic_frame.groupby("location_id"):
        axes[0, 1].scatter(
            location_data["timestamp"],
            location_data["data_value"],
            label=f"Location {location_id}",
            s=18,
        )
    axes[0, 1].set_title("Synthetic traffic-speed readings")
    axes[0, 1].set_xlabel("Simulation time")
    axes[0, 1].set_ylabel("Speed (km/h)")
    axes[0, 1].tick_params(axis="x", rotation=30)
    axes[0, 1].legend(title="Traffic location", fontsize="small")

    if digital_twin.disruption_log:
        event_times = [event.detection_time for event in digital_twin.disruption_log]
        event_severities = [event.severity for event in digital_twin.disruption_log]
        axes[1, 0].scatter(event_times, event_severities, label="Detected event")
    axes[1, 0].axhline(
        y=digital_twin.config.disruption_threshold,
        color="red",
        linestyle="--",
        label="Severity threshold",
    )
    axes[1, 0].set_title("Threshold-based disruption detections")
    axes[1, 0].set_xlabel("Detection time")
    axes[1, 0].set_ylabel("Severity (dimensionless)")
    axes[1, 0].tick_params(axis="x", rotation=30)
    axes[1, 0].legend()

    if digital_twin.disruption_log:
        location_counts = (
            pd.Series([event.location for event in digital_twin.disruption_log])
            .value_counts()
            .reindex(range(1, 8), fill_value=0)
            .sort_index()
        )
    else:
        location_counts = pd.Series(0, index=range(1, 8))
    axes[1, 1].bar(location_counts.index, location_counts.values)
    axes[1, 1].set_title("Detected events by traffic location")
    axes[1, 1].set_xlabel("Traffic location")
    axes[1, 1].set_ylabel("Event count")
    axes[1, 1].set_xticks(range(1, 8))

    fig.suptitle(
        "Synthetic Telemetry Simulation "
        f"({simulation_results['duration_minutes']} simulated minutes)",
        fontsize=16,
        fontweight="bold",
    )
    plt.tight_layout()
    plt.show()


visualize_digital_twin_performance(digital_twin, simulation_results)

## 10. Summary and interpretation

This notebook demonstrates a small data-flow pattern relevant to digital-twin systems: generate observations, apply a transparent detection rule, retain events, and inspect the resulting telemetry. The event detector's output is reproducible in its sensor values when the notebook is run from the top, while simulation timestamps depend on the runtime clock.

The VRPB instance provides routing context: delivery customers, pickup customers, demands, vehicle capacity, and pairwise distances. In a VRPB route, delivery-before-pickup precedence is central because the vehicle carries delivery goods at departure and collects backhauls after deliveries. No route is constructed here, so feasibility, total distance, and precedence compliance are not evaluated by the code.

### What the results do and do not mean

- The processed-data count is the number of synthetic observations actually generated by the loops.
- A disruption count is the number of traffic observations that meet the detector's speed and severity conditions.
- The plotted distributions and event locations describe this simulation run; they are not measured operational performance.
- The configured 50,000-points-per-hour figure is not achieved or tested by the sample generator.
- The duration and impact fields are rule-based illustrative estimates. They do not establish predictive accuracy, delay reduction, or cost savings.
- The simulation does not integrate live sensors, predict future traffic, reassign pickups, or preserve route constraints during dynamic reoptimization.

### Further modeling considerations

A fuller operational study would need validated data, explicit links between traffic locations and route arcs, a route plan to evaluate, and a defined reoptimization procedure. Those elements are outside this notebook's implemented scope. The central VRPB feature to preserve in any route plan remains that each vehicle serves its assigned deliveries before its assigned pickups.